In [9]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
import datetime
import itertools
import matplotlib.dates as mdates
import pandas as pd
import gc
import os



# Clear Keras models/layers from memory
keras.backend.clear_session()

# Force Python garbage collection
gc.collect()

# Optional: reset TF's internal state further
tf.keras.backend.clear_session()
gc.collect()

import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

memory.used [MiB], memory.total [MiB]
8386 MiB, 10240 MiB
325 MiB, 10240 MiB



In [10]:
# Data path from AGENTS.md
data_folder = "/home/haseebumer/content/data/extracted/"

# Model identifier and output directory
model_name = "CNN_LSTM_forecast_15m_30m_1h_2017_2019_data"
output_folder = os.path.join("model_output", model_name)

# Ensure output directory exists
os.makedirs(output_folder, exist_ok=True)

# File paths for extracted datasets
trainval_images_path = os.path.join(data_folder, "trainval_images_log.npy")
trainval_pv_path = os.path.join(data_folder, "trainval_pv_log.npy")
test_images_path = os.path.join(data_folder, "test_images_log.npy")
test_pv_path = os.path.join(data_folder, "test_pv_log.npy")

times_trainval_path = os.path.join(data_folder, "times_trainval.npy")
times_test_path = os.path.join(data_folder, "times_test.npy")

print("data_folder:   ", data_folder)
print("output_folder: ", output_folder)
print("trainval_images:", trainval_images_path)
print("trainval_pv:   ", trainval_pv_path)
print("test_images:    ", test_images_path)
print("test_pv:       ", test_pv_path)
print("times_trainval:", times_trainval_path)
print("times_test:    ", times_test_path)

data_folder:    /home/haseebumer/content/data/extracted/
output_folder:  model_output/CNN_LSTM_forecast_15m_30m_1h_2017_2019_data
trainval_images: /home/haseebumer/content/data/extracted/trainval_images_log.npy
trainval_pv:    /home/haseebumer/content/data/extracted/trainval_pv_log.npy
test_images:     /home/haseebumer/content/data/extracted/test_images_log.npy
test_pv:        /home/haseebumer/content/data/extracted/test_pv_log.npy
times_trainval: /home/haseebumer/content/data/extracted/times_trainval.npy
times_test:     /home/haseebumer/content/data/extracted/times_test.npy


In [11]:
# Model & Forecast Horizon Parameters
# SEQ_LEN: Number of historical 1-minute time steps used as input (16 minutes)
SEQ_LEN = 16

FORECAST_HORIZONS = [15, 30, 60]
HORIZON_NAMES = ["15 min", "30 min", "1 hour"]
NUM_TARGETS = len(FORECAST_HORIZONS)

# Training hyper-parameters
NUM_FOLDS = 10  # 10-fold cross validation
NUM_EPOCHS = 200  # Max epochs (EarlyStopping will halt earlier)
BATCH_SIZE = 16
LEARNING_RATE = 1e-3
PATIENCE = 10  # Early stopping patience

print(f"Sequence Length (History Window): {SEQ_LEN} minutes")
print(f"Target Horizons:                  {FORECAST_HORIZONS} steps ({HORIZON_NAMES})")
print(f"Number of Target Outputs:         {NUM_TARGETS}")
print(f"Cross-Validation Folds:           {NUM_FOLDS}")

Sequence Length (History Window): 16 minutes
Target Horizons:                  [15, 30, 60] steps (['15 min', '30 min', '1 hour'])
Number of Target Outputs:         3
Cross-Validation Folds:           10


In [12]:
# Memory-map large image files to prevent RAM overflow / kernel crashes
trainval_images = np.load(trainval_images_path, mmap_mode="r")
test_images = np.load(test_images_path, mmap_mode="r")

trainval_pv = np.load(trainval_pv_path).astype(np.float32)
test_pv = np.load(test_pv_path).astype(np.float32)

times_trainval = np.load(times_trainval_path, allow_pickle=True)
times_test = np.load(times_test_path, allow_pickle=True)

print(f"trainval_images shape: {trainval_images.shape} | dtype: {trainval_images.dtype}")
print(f"test_images shape:     {test_images.shape} | dtype: {test_images.dtype}")
print(f"trainval_pv shape:     {trainval_pv.shape} | times: {times_trainval.shape}")
print(f"test_pv shape:         {test_pv.shape} | times: {times_test.shape}")

trainval_images shape: (349372, 64, 64, 3) | dtype: uint8
test_images shape:     (14003, 64, 64, 3) | dtype: uint8
trainval_pv shape:     (349372,) | times: (349372,)
test_pv shape:         (14003,) | times: (14003,)


In [13]:
def create_day_aware_forecast_sequences(
    pv_array, times_array, images_array=None, seq_len=16, horizons=[15, 30, 60], allow_partial=False
):
    """
    Constructs (X, y, sequence_dates, sequence_times) sequences
    ensuring that no sequence crosses overnight boundaries or distinct calendar dates.
    To prevent RAM overflow and kernel crashes, images are not duplicated into RAM;
    instead, integer indices are tracked to stream images on-demand from disk/mmap.

    Parameters:
        pv_array: 1D numpy array of PV power generation values (kW)
        times_array: 1D numpy array of timestamps (datetime objects)
        images_array: mmap-loaded array of sky camera images (N, 64, 64, 3) or None
        seq_len: Number of historical lag steps for input X
        horizons: List/tuple of forecast steps ahead [15, 30, 60]
        allow_partial: If True, generates sequences all the way to sunset
                       (cutoff at min(horizons) instead of max(horizons)).
                       Horizons that exceed sunset are padded with NaN / None.
                       This allows 15m and 30m forecasts to run through the entire afternoon!

    Returns:
        X: Multimodal bundle (images_array, X_pv, img_indices) if images_array is provided,
           otherwise X_pv of shape (N, seq_len, 1)
        y: Array of shape (N, len(horizons)) [or (N,) if single horizon]
        valid_dates: Array of base calendar dates (N,)
        valid_times: Array of target forecast timestamps (N, len(horizons))
    """
    is_multi = isinstance(horizons, (list, tuple, np.ndarray))
    if is_multi:
        horizon_list = list(horizons)
        max_horizon = max(horizon_list)
        min_horizon = min(horizon_list)
    else:
        horizon_list = [horizons]
        max_horizon = horizons
        min_horizon = horizons

    cutoff_horizon = min_horizon if allow_partial else max_horizon

    dates = np.array(
        [
            (
                t.date()
                if isinstance(t, (datetime.datetime, datetime.date))
                else pd.to_datetime(t).date()
            )
            for t in times_array
        ]
    )

    unique_dates = np.unique(dates)

    X_pv_list, y_list = [], []
    valid_dates_list = []
    valid_times_list = []
    img_indices_list = [] if images_array is not None else None

    for d in unique_dates:
        day_indices = np.where(dates == d)[0]
        num_day_points = len(day_indices)

        # Skip days that do not have enough points for sequence + cutoff horizon
        if num_day_points <= seq_len + cutoff_horizon:
            continue

        day_pv = pv_array[day_indices]
        day_times = times_array[day_indices]

        for i in range(seq_len, num_day_points - cutoff_horizon):
            seq_x = day_pv[i - seq_len : i]

            if is_multi:
                seq_y = []
                target_times = []
                for h in horizon_list:
                    if i + h < num_day_points:
                        seq_y.append(day_pv[i + h])
                        target_times.append(day_times[i + h])
                    else:
                        seq_y.append(np.nan)
                        target_times.append(None)
            else:
                seq_y = day_pv[i + horizon_list[0]]
                target_times = day_times[i + horizon_list[0]]

            X_pv_list.append(seq_x)
            y_list.append(seq_y)
            valid_dates_list.append(d)
            valid_times_list.append(target_times)

            if images_array is not None:
                # Image indices aligned to the PV history window: t-seq_len ... t-1
                img_indices_list.append(day_indices[i - seq_len : i])

    X_pv = np.expand_dims(
        np.array(X_pv_list, dtype=np.float32), axis=-1
    )  # Shape: (N, seq_len, 1)

    y = np.array(y_list, dtype=np.float32)  # Shape: (N, num_horizons) or (N,)

    valid_dates = np.array(valid_dates_list)
    valid_times = np.array(valid_times_list, dtype=object)

    if images_array is not None:
        img_indices = np.array(img_indices_list, dtype=np.int32)
        # Return multimodal bundle: (mmap_images, pv_sequences, image_indices)
        return (images_array, X_pv, img_indices), y, valid_dates, valid_times
    else:
        return X_pv, y, valid_dates, valid_times


X_trainval, y_trainval, dates_trainval_seq, times_trainval_seq = (
    create_day_aware_forecast_sequences(
        trainval_pv,
        times_trainval,
        images_array=trainval_images,
        seq_len=SEQ_LEN,
        horizons=FORECAST_HORIZONS,
        allow_partial=False,
    )
)

# Set allow_partial=True for test set so 15m and 30m forecasts continue all the way to sunset!
X_test, y_test, dates_test_seq, times_test_seq = create_day_aware_forecast_sequences(
    test_pv,
    times_test,
    images_array=test_images,
    seq_len=SEQ_LEN,
    horizons=FORECAST_HORIZONS,
    allow_partial=True,
)

print(
    f"X_trainval sequences: {len(y_trainval):,} | "
    f"PV shape: {X_trainval[1].shape} | "
    f"y_trainval shape: {y_trainval.shape} | "
    f"Images storage: {X_trainval[0].shape} (mmap on-demand)"
)
print(
    f"X_test sequences:     {len(y_test):,} | "
    f"PV shape: {X_test[1].shape} | "
    f"y_test shape: {y_test.shape} | "
    f"Images storage: {X_test[0].shape} (mmap on-demand)"
)
print(
    f"dates_trainval:       {dates_trainval_seq.shape}, times_trainval: {times_trainval_seq.shape}"
)
print(
    f"dates_test:           {dates_test_seq.shape}, times_test:       {times_test_seq.shape}"
)

X_trainval sequences: 311,600 | PV shape: (311600, 16, 1) | y_trainval shape: (311600, 3) | Images storage: (349372, 64, 64, 3) (mmap on-demand)
X_test sequences:     13,383 | PV shape: (13383, 16, 1) | y_test shape: (13383, 3) | Images storage: (14003, 64, 64, 3) (mmap on-demand)
dates_trainval:       (311600,), times_trainval: (311600, 3)
dates_test:           (13383,), times_test:       (13383, 3)


In [14]:
def day_block_shuffle_indices(dates_array, seed=1):
    """
    Returns unique calendar dates in shuffled order.
    """
    unique_dates = np.unique(dates_array)
    rng = np.random.RandomState(seed)
    shuffled_dates = unique_dates.copy()
    rng.shuffle(shuffled_dates)
    return shuffled_dates


def cv_split_indices(shuffled_dates, dates_array, fold_index, num_folds=10):
    """
    Splits shuffled dates into num_folds contiguous date-blocks.
    fold_index's block becomes validation; every sample belonging
    to a date in that block goes to validation, regardless of how
    many sequences that day contributed — no day is ever split
    across train/val.
    """
    date_folds = np.array_split(shuffled_dates, num_folds)
    val_dates = set(date_folds[fold_index])

    val_mask = np.isin(dates_array, list(val_dates))

    all_idx = np.arange(len(dates_array))
    val_idx = all_idx[val_mask]
    train_idx = all_idx[~val_mask]

    rng = np.random.RandomState(fold_index)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)

    return train_idx, val_idx


def build_tf_dataset(X, y, indices, batch_size=16, is_training=True):
    """
    Builds an optimized streaming tf.data.Dataset pipeline.
    Streams image batches on-demand directly from memory-mapped disk storage,
    preventing RAM exhaustion (OOM) and kernel crashes.
    Supports multi-horizon forecasting targets.
    Uses prefetching and AUTOTUNE for maximum GPU training throughput.
    """
    if isinstance(X, (tuple, list)) and len(X) == 3:
        images_mmap, X_pv, img_indices = X
        sub_indices = np.array(indices)
        num_samples = len(sub_indices)
        num_batches = int(np.ceil(num_samples / batch_size))

        def generator():
            curr_idx = sub_indices.copy()
            if is_training:
                np.random.shuffle(curr_idx)
            for start in range(0, num_samples, batch_size):
                b_idx = curr_idx[start : start + batch_size]
                b_img_idx = img_indices[b_idx]          # now shape (batch, seq_len)
                b_img = images_mmap[b_img_idx]           # fancy-indexes to (batch, seq_len, 64, 64, 3)
                b_pv = X_pv[b_idx]
                b_y = y[b_idx]
                yield (b_img, b_pv), b_y

        target_shape = (None, y.shape[1]) if y.ndim > 1 else (None,)

        output_signature = (
            (
                tf.TensorSpec(shape=(None, X_pv.shape[1], 64, 64, 3), dtype=tf.uint8),
                tf.TensorSpec(
                    shape=(None, X_pv.shape[1], X_pv.shape[2]), dtype=tf.float32
                ),
            ),
            tf.TensorSpec(shape=target_shape, dtype=tf.float32),
        )

        ds = tf.data.Dataset.from_generator(
            generator, output_signature=output_signature
        )
        ds = ds.apply(tf.data.experimental.assert_cardinality(num_batches))
        return ds.prefetch(tf.data.AUTOTUNE)
    else:
        X_sub = X[indices]
        y_sub = y[indices]

        ds = tf.data.Dataset.from_tensor_slices((X_sub, y_sub))

        if is_training:
            ds = ds.shuffle(buffer_size=min(len(indices), 10000), seed=42)

        ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

        return ds

In [15]:
def build_cnn_feature_extractor(
    input_shape=(64, 64, 3),
    filters=(32, 64, 128),
    feature_dim=64,
    name="cnn_feature_extractor",
):
    """
    Builds a 2D CNN backbone to extract visual features from sky camera images.
    Uses tf.keras.layers.Rescaling(1.0 / 255.0) to normalize uint8 images on-GPU.
    """
    img_in = keras.Input(shape=input_shape, name="cnn_image_in")
    x = keras.layers.Rescaling(1.0 / 255.0, name="rescaling")(img_in)
    for i, f in enumerate(filters):
        x = keras.layers.Conv2D(
            filters=f,
            kernel_size=(3, 3),
            padding="same",
            activation="relu",
            name=f"conv_{i+1}",
        )(x)
        x = keras.layers.BatchNormalization(name=f"bn_{i+1}")(x)
        x = keras.layers.MaxPooling2D(pool_size=(2, 2), name=f"pool_{i+1}")(x)

    # Spatial pooling into a dense feature vector
    x = keras.layers.GlobalAveragePooling2D(name="gap")(x)
    x = keras.layers.Dense(feature_dim, activation="relu", name="cnn_feature_proj")(x)

    return keras.Model(inputs=img_in, outputs=x, name=name)


def build_cnn_lstm_forecast_model(
    seq_len=16,
    num_features=1,
    num_outputs=3,
    image_shape=(SEQ_LEN, 64, 64, 3),
    cnn_filters=(32, 64),
    cnn_feature_dim=64,
    lstm_units=(64, 64),
    dense_units=128,
):
    """
    Builds a Multimodal CNN + LSTM Fusion model for multi-horizon solar forecasting:
    1. Visual features from concurrent sky camera image are extracted via 2D CNN backbone.
    2. Image features are projected and repeated across time steps (seq_len).
    3. Image features are concatenated with the historical time series PV sequence.
    4. The combined representation is passed to Stacked LSTM layers for temporal modeling.
    5. A Dense regression head outputs multi-horizon PV forecast predictions ([15m, 30m, 1h]).
    """
    is_image_sequence = len(image_shape) == 4

    if is_image_sequence:
        image_input = keras.Input(shape=image_shape, name="image_sequence_input")
        single_img_shape = image_shape[1:]
    else:
        image_input = keras.Input(shape=image_shape, name="image_input")
        single_img_shape = image_shape

    pv_input = keras.Input(shape=(seq_len, num_features), name="pv_sequence_input")

    # 2. Extract image features via CNN
    cnn_extractor = build_cnn_feature_extractor(
        input_shape=single_img_shape,
        filters=cnn_filters,
        feature_dim=cnn_feature_dim,
        name="cnn_backbone",
    )

    if is_image_sequence:
        img_features_seq = keras.layers.TimeDistributed(
            cnn_extractor, name="time_distributed_cnn"
        )(image_input)
    else:
        img_features = cnn_extractor(image_input)
        # Repeat single image features across time steps to align with the PV sequence
        img_features_seq = keras.layers.RepeatVector(
            seq_len, name="repeat_image_features"
        )(img_features)

    # 3. Concatenate image features with corresponding time series PV data
    fused = keras.layers.Concatenate(axis=-1, name="fusion_concat")(
        [img_features_seq, pv_input]
    )

    # 4. Pass fused sequence to LSTM
    x = fused
    if isinstance(lstm_units, (list, tuple)):
        for i, units in enumerate(lstm_units):
            is_last = i == len(lstm_units) - 1
            x = keras.layers.LSTM(
                units=units,
                return_sequences=not is_last,
                name=f"lstm_{i+1}",
            )(x)
    else:
        x = keras.layers.LSTM(
            units=lstm_units,
            return_sequences=False,
            name="lstm_1",
        )(x)

    # 5. Dense head for multi-horizon forecast prediction
    x = keras.layers.Dense(dense_units, activation="relu", name="dense_1")(x)

    outputs = keras.layers.Dense(
        units=num_outputs, activation="linear", name="pv_forecast"
    )(x)

    model = keras.Model(
        inputs=[image_input, pv_input],
        outputs=outputs,
        name=f"CNN_LSTM_Solar_Forecaster_{num_outputs}Horizons",
    )

    return model


sample_model = build_cnn_lstm_forecast_model(
    seq_len=SEQ_LEN,
    num_features=1,
    num_outputs=NUM_TARGETS,
    image_shape=(SEQ_LEN, 64, 64, 3),
)

sample_model.summary()

Model: "CNN_LSTM_Solar_Forecaster_3Horizons"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_sequence_inp… │ (None, 16, 64,    │          0 │ -                 │
│ (InputLayer)        │ 64, 3)            │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_c… │ (None, 16, 64)    │     23,936 │ image_sequence_i… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pv_sequence_input   │ (None, 16, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fusion_concat       │ (None, 16, 65)    │          0 │ time_distributed… │
│ (Concatenate)       │                   │            │ pv_sequence_inpu… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 16, 64)    │     33,280 │ fusion_concat[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ (None, 64)        │     33,024 │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │      8,320 │ lstm_2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pv_forecast (Dense) │ (None, 3)         │        387 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 98,947 (386.51 KB)

 Trainable params: 98,755 (385.76 KB)

 Non-trainable params: 192 (768.00 B)

In [ ]:
shuffled_dates = day_block_shuffle_indices(dates_trainval_seq, seed=1)

# --- Resume support: load existing history if this cell was run before ---
history_paths = {
    "train_loss": os.path.join(output_folder, "train_loss_hist.npy"),
    "val_loss": os.path.join(output_folder, "val_loss_hist.npy"),
    "train_mae": os.path.join(output_folder, "train_mae_hist.npy"),
    "val_mae": os.path.join(output_folder, "val_mae_hist.npy"),
}

if all(os.path.exists(p) for p in history_paths.values()):
    train_loss_hist = list(np.load(history_paths["train_loss"], allow_pickle=True))
    val_loss_hist = list(np.load(history_paths["val_loss"], allow_pickle=True))
    train_mae_hist = list(np.load(history_paths["train_mae"], allow_pickle=True))
    val_mae_hist = list(np.load(history_paths["val_mae"], allow_pickle=True))
    print(
        f"Resuming: found existing history for {len(train_loss_hist)} completed fold(s)."
    )
else:
    train_loss_hist = []
    val_loss_hist = []
    train_mae_hist = []
    val_mae_hist = []

# A fold counts as "done" only if its history was already recorded above
start_fold = len(train_loss_hist)

if start_fold >= NUM_FOLDS:
    print(f"All {NUM_FOLDS} folds already completed. Nothing to train.")

for fold in range(start_fold, NUM_FOLDS):
    print(f"\n{'='*25} FOLD {fold + 1} / {NUM_FOLDS} {'='*25}")

    train_idx, val_idx = cv_split_indices(
        shuffled_dates, dates_trainval_seq, fold_index=fold, num_folds=NUM_FOLDS
    )

    # Datasets
    ds_train = build_tf_dataset(
        X_trainval, y_trainval, train_idx, batch_size=BATCH_SIZE, is_training=True
    )

    ds_val = build_tf_dataset(
        X_trainval, y_trainval, val_idx, batch_size=BATCH_SIZE, is_training=False
    )

    # Model definition
    keras.backend.clear_session()

    model = build_cnn_lstm_forecast_model(
        seq_len=SEQ_LEN,
        num_features=1,
        num_outputs=NUM_TARGETS,
        image_shape=(SEQ_LEN, 64, 64, 3),
        cnn_filters=(32, 64),
        cnn_feature_dim=64,
        lstm_units=(64, 64),
        dense_units=128,
    )

    optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE)
    model.compile(optimizer=optimizer, loss="mse", metrics=["mae"])

    # Checkpoint and Logging directories
    checkpoint_dir = os.path.join(output_folder, f"repetition_{fold + 1}")
    os.makedirs(checkpoint_dir, exist_ok=True)
    checkpoint_path = os.path.join(
        checkpoint_dir, f"best_model_repetition_{fold + 1}.h5"
    )

    tensorboard_log_dir = os.path.join(
        output_folder, "tensorboard_logs", f"repetition_{fold + 1}"
    )
    os.makedirs(tensorboard_log_dir, exist_ok=True)

    # Callbacks
    earlystop = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        restore_best_weights=True,
        mode="min",
        verbose=1,
    )

    checkpoint = keras.callbacks.ModelCheckpoint(
        checkpoint_path, monitor="val_loss", mode="min", save_best_only=True, verbose=1
    )

    reduce_lr = keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    )

    tensorboard = keras.callbacks.TensorBoard(
        log_dir=tensorboard_log_dir,
        histogram_freq=1,
        write_graph=True,
        update_freq="epoch",
    )

    # Training
    history = model.fit(
        ds_train,
        epochs=NUM_EPOCHS,
        validation_data=ds_val,
        callbacks=[earlystop, checkpoint, reduce_lr, tensorboard],
        verbose=1,
    )

    # Store training history
    train_loss_hist.append(history.history["loss"])
    val_loss_hist.append(history.history["val_loss"])
    train_mae_hist.append(history.history["mae"])
    val_mae_hist.append(history.history["val_mae"])

    # Save history arrays after EVERY fold for crash resilience
    np.save(history_paths["train_loss"], np.array(train_loss_hist, dtype=object))
    np.save(history_paths["val_loss"], np.array(val_loss_hist, dtype=object))
    np.save(history_paths["train_mae"], np.array(train_mae_hist, dtype=object))
    np.save(history_paths["val_mae"], np.array(val_mae_hist, dtype=object))

    print(f"Fold {fold + 1} complete and saved.")


========================= FOLD 1 / 10 =========================
Epoch 1/200


/home/haseebumer/skippd-solar-forecasting/aienv/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


 1617/17627 ━━━━━━━━━━━━━━━━━━━━ 8:53 33ms/step - loss: 11.2886 - mae: 1.8948

In [ ]:
fig, axarr = plt.subplots(1, 2, figsize=(16, 6))

# TRAINING & VALIDATION LOSS
for fold in range(NUM_FOLDS):
    axarr[0].plot(train_loss_hist[fold], label=f"Fold {fold + 1} Train")
    axarr[0].plot(
        val_loss_hist[fold], linestyle="--", label=f"Fold {fold + 1} Validation"
    )

axarr[0].set_xlabel("Epoch")
axarr[0].set_ylabel("MSE Loss")
axarr[0].set_title("Training and Validation Loss Across Folds")
axarr[0].legend(ncol=2, fontsize=8)
axarr[0].grid(True)

# TRAINING & VALIDATION MAE
for fold in range(NUM_FOLDS):
    axarr[1].plot(train_mae_hist[fold], label=f"Fold {fold + 1} Train")
    axarr[1].plot(
        val_mae_hist[fold], linestyle="--", label=f"Fold {fold + 1} Validation"
    )

axarr[1].set_xlabel("Epoch")
axarr[1].set_ylabel("MAE (kW)")
axarr[1].set_title("Training and Validation MAE Across Folds")
axarr[1].legend(ncol=2, fontsize=8)
axarr[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
best_train_loss_mse = np.zeros(NUM_FOLDS)
best_val_loss_mse = np.zeros(NUM_FOLDS)

print(f"{'Fold':<8} | {'Best Train RMSE (kW)':<22} | {'Best Val RMSE (kW)':<20}")
print("-" * 56)

for i in range(NUM_FOLDS):
    best_val_loss_mse[i] = np.min(val_loss_hist[i])
    idx = np.argmin(val_loss_hist[i])
    best_train_loss_mse[i] = train_loss_hist[i][idx]

    train_rmse = np.sqrt(best_train_loss_mse[i])
    val_rmse = np.sqrt(best_val_loss_mse[i])
    print(f"Fold {i + 1:02d}  | {train_rmse:<22.3f} | {val_rmse:<20.3f}")

print("-" * 56)
print(f"Mean Train RMSE:      {np.mean(np.sqrt(best_train_loss_mse)):.3f} kW")
print(f"Mean Validation RMSE: {np.mean(np.sqrt(best_val_loss_mse)):.3f} kW")

In [ ]:
predictions = np.zeros((NUM_FOLDS, len(y_test), NUM_TARGETS), dtype=np.float32)

# Build streaming test dataset (batch_size=512 for high-throughput GPU inference)
ds_test = build_tf_dataset(
    X_test, y_test, np.arange(len(y_test)), batch_size=512, is_training=False
)

for fold in range(NUM_FOLDS):
    checkpoint_path = os.path.join(
        output_folder, f"repetition_{fold + 1}", f"best_model_repetition_{fold + 1}.h5"
    )

    print(f"Loading fold {fold + 1} model from: {checkpoint_path}")
    fold_model = build_cnn_lstm_forecast_model(
        seq_len=SEQ_LEN,
        num_features=1,
        num_outputs=NUM_TARGETS,
        image_shape=(SEQ_LEN, 64, 64, 3),
    )
    fold_model.load_weights(checkpoint_path)

    pred = fold_model.predict(ds_test, verbose=0)
    if pred.ndim == 1 or (pred.ndim == 2 and pred.shape[1] == 1 and NUM_TARGETS == 1):
        pred = np.reshape(pred, (-1, 1))

    predictions[fold] = pred

# Save individual fold predictions
np.save(os.path.join(output_folder, "test_predictions_all_folds.npy"), predictions)

# Ensemble average across folds
ensemble_predictions = np.mean(predictions, axis=0)

np.save(
    os.path.join(output_folder, "test_predictions_ensemble.npy"), ensemble_predictions
)
print(f"\nEnsemble predictions saved. Shape: {ensemble_predictions.shape}")

In [ ]:
# TEST SET EVALUATION ACROSS ALL HORIZONS
errors = ensemble_predictions - y_test  # Shape: (N, NUM_TARGETS)

print("=" * 68)
print(f"COMPLETE TEST SET FORECAST RESULTS ({len(y_test):,} Sequences)")
print("=" * 68)
print(
    f"{'Horizon':<12} | {'Lead Steps':<12} | {'Test RMSE (kW)':<16} | {'Test MAE (kW)':<14}"
)
print("-" * 68)

horizon_metrics = {}
for k, (h_name, h_steps) in enumerate(zip(HORIZON_NAMES, FORECAST_HORIZONS)):
    h_err = errors[:, k] if errors.ndim > 1 else errors
    valid = ~np.isnan(h_err)
    h_rmse = np.sqrt(np.mean(np.square(h_err[valid])))
    h_mae = np.mean(np.abs(h_err[valid]))
    horizon_metrics[h_name] = {"rmse": h_rmse, "mae": h_mae}
    print(f"{h_name:<12} | {h_steps:<12} | {h_rmse:<16.3f} | {h_mae:<14.3f}")

print("-" * 68)
valid_all = ~np.isnan(errors)
rmse_overall = np.sqrt(np.mean(np.square(errors[valid_all])))
mae_overall = np.mean(np.abs(errors[valid_all]))
print(f"{'OVERALL':<12} | {'All':<12} | {rmse_overall:<16.3f} | {mae_overall:<14.3f}")
print("=" * 68)

# TIME-SERIES PLOTS FOR COMPLETE TEST SET
fig, axes = plt.subplots(NUM_TARGETS, 1, figsize=(18, 5 * NUM_TARGETS), sharex=True)
if NUM_TARGETS == 1:
    axes = [axes]

colors = ["#1f77b4", "#2ca02c", "#030302"]  # Blue (15m), Green (30m), Black (1h)

for k, (h_name, h_color) in enumerate(zip(HORIZON_NAMES, colors)):
    ax = axes[k]
    h_times = times_test_seq[:, k] if times_test_seq.ndim > 1 else times_test_seq
    h_y_true = y_test[:, k] if y_test.ndim > 1 else y_test
    h_y_pred = (
        ensemble_predictions[:, k]
        if ensemble_predictions.ndim > 1
        else ensemble_predictions
    )
    valid_k = np.array(
        [t is not None and not np.isnan(y) for t, y in zip(h_times, h_y_true)]
    )

    ax.plot(
        h_times[valid_k],
        h_y_true[valid_k],
        color="#808080",
        alpha=0.6,
        label="Actual PV",
    )
    ax.plot(
        h_times[valid_k],
        h_y_pred[valid_k],
        color=h_color,
        alpha=0.85,
        label=f"CNN-LSTM Forecast ({h_name})",
    )
    ax.set_ylabel("PV Power (kW)", fontsize=11)
    ax.set_title(
        f"Complete Test Set: Actual vs {h_name} Forecast "
        f"(RMSE: {horizon_metrics[h_name]['rmse']:.3f} kW | MAE: {horizon_metrics[h_name]['mae']:.3f} kW)",
        fontsize=12,
        fontweight="bold",
    )
    ax.legend(loc="upper right")
    ax.grid(True)

axes[-1].set_xlabel("Time", fontsize=11)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# SUNNY AND CLOUDY DATES FROM THE SKIPP'D PAPER
sunny_dates = [
    (2017, 9, 15),
    (2017, 10, 6),
    (2017, 10, 22),
    (2018, 2, 16),
    (2018, 6, 12),
    (2018, 6, 23),
    (2019, 1, 25),
    (2019, 6, 23),
    (2019, 7, 14),
    (2019, 10, 14),
]

cloudy_dates = [
    (2017, 6, 24),
    (2017, 9, 20),
    (2017, 10, 11),
    (2018, 1, 25),
    (2018, 3, 9),
    (2018, 10, 4),
    (2019, 5, 27),
    (2019, 6, 28),
    (2019, 8, 10),
    (2019, 10, 19),
]

sunny_dates_test = [datetime.date(y, m, d) for y, m, d in sunny_dates]
cloudy_dates_test = [datetime.date(y, m, d) for y, m, d in cloudy_dates]

In [ ]:
import math

# Precompute full-day test dates to plot complete diurnal ground truth
dates_test_all = np.array(
    [
        (
            t.date()
            if isinstance(t, (datetime.datetime, datetime.date))
            else pd.to_datetime(t).date()
        )
        for t in times_test
    ]
)

xfmt = mdates.DateFormatter("%H")
fmt_date = datetime.date(2000, 1, 1)

COLOR_GT = "#B6B1A9"
HORIZON_COLORS = [
    "#1f77b4",
    "#2ca02c",
    "#ff7f0e",
]  # Blue (15m), Green (30m), Orange (1h)


def plot_single_horizon(
    ax, target_date, day_type_label, horizon_idx, show_legend=False
):
    """Plot full-day ground truth + a SINGLE horizon's forecast on one axis."""
    date_mask = dates_test_seq == target_date
    if not np.any(date_mask):
        ax.text(
            0.5, 0.5, f"Date {target_date}\nnot in test set", ha="center", fontsize=7
        )
        return

    h_name = HORIZON_NAMES[horizon_idx]
    h_color = HORIZON_COLORS[horizon_idx]

    # --- Full-day ground truth (sunrise to sunset) ---
    day_mask_full = dates_test_all == target_date
    if np.any(day_mask_full):
        times_day_full = times_test[day_mask_full]
        y_gt = test_pv[day_mask_full]
    else:
        times_day_full = (
            times_test_seq[date_mask, 0]
            if times_test_seq.ndim > 1
            else times_test_seq[date_mask]
        )
        y_gt = y_test[date_mask, 0] if y_test.ndim > 1 else y_test[date_mask]

    hours_xaxis_gt = [
        datetime.datetime.combine(
            fmt_date,
            (
                t.time()
                if isinstance(t, (datetime.datetime, datetime.time))
                else pd.to_datetime(t).time()
            ),
        )
        for t in times_day_full
    ]

    ax.plot(
        hours_xaxis_gt,
        y_gt,
        color=COLOR_GT,
        linewidth=1.2,
        label="Ground truth" if show_legend else "",
    )
    ax.fill_between(hours_xaxis_gt, 0, y_gt, color=COLOR_GT, alpha=0.25)

    # --- Single horizon forecast ---
    y_true_day = y_test[date_mask]
    y_pred_day = ensemble_predictions[date_mask]

    times_day_hk = (
        times_test_seq[date_mask, horizon_idx]
        if times_test_seq.ndim > 1
        else times_test_seq[date_mask]
    )
    valid_mask = np.array([t is not None for t in times_day_hk])

    if np.any(valid_mask):
        valid_times_k = times_day_hk[valid_mask]
        hours_xaxis_k = [
            datetime.datetime.combine(
                fmt_date,
                (
                    t.time()
                    if isinstance(t, (datetime.datetime, datetime.time))
                    else pd.to_datetime(t).time()
                ),
            )
            for t in valid_times_k
        ]

        y_pred_k = (
            y_pred_day[valid_mask, horizon_idx]
            if y_pred_day.ndim > 1
            else y_pred_day[valid_mask]
        )
        y_true_k = (
            y_true_day[valid_mask, horizon_idx]
            if y_true_day.ndim > 1
            else y_true_day[valid_mask]
        )

        metric_valid = ~np.isnan(y_true_k)
        metric_text = ""
        if np.any(metric_valid):
            d_rmse = np.sqrt(
                np.mean(np.square(y_true_k[metric_valid] - y_pred_k[metric_valid]))
            )
            d_mae = np.mean(np.abs(y_true_k[metric_valid] - y_pred_k[metric_valid]))
            metric_text = f"RMSE {d_rmse:.2f}\nMAE {d_mae:.2f}"

        ax.plot(
            hours_xaxis_k,
            y_pred_k,
            color=h_color,
            linewidth=1.3,
            label=f"CNN-LSTM {h_name}" if show_legend else "",
        )

        if metric_text:
            ax.text(
                0.03,
                0.90,
                metric_text,
                transform=ax.transAxes,
                fontsize=6.5,
                va="top",
            )

    ax.xaxis.set_major_formatter(xfmt)
    ax.set_ylim(0, 30)
    ax.set_title(
        f"{day_type_label} — {target_date}\n{h_name} forecast",
        fontsize=7.5,
        fontweight="bold",
    )


# ------------------------------------------------------------------
# Build grid: rows = max(n_sunny, n_cloudy)
# Cols 0-2: Sunny  -> 15m / 30m / 1h
# Cols 3-5: Cloudy -> 15m / 30m / 1h
# ------------------------------------------------------------------
n_sunny = len(sunny_dates_test)
n_cloudy = len(cloudy_dates_test)
n_rows = max(n_sunny, n_cloudy)

fig, axarr = plt.subplots(n_rows, 6, figsize=(22, 2.6 * n_rows), sharey=True)

if n_rows == 1:
    axarr = axarr.reshape(1, -1)

# Turn everything off by default
for r in range(n_rows):
    for c in range(6):
        axarr[r, c].axis("off")

first_legend_done = False

for i in range(n_rows):
    # --- Sunny row i -> columns 0,1,2 ---
    if i < n_sunny:
        date = sunny_dates_test[i]
        for k in range(3):
            ax = axarr[i, k]
            ax.axis("on")
            show_legend = not first_legend_done
            plot_single_horizon(ax, date, "Sunny", k, show_legend=show_legend)
        first_legend_done = (
            True  # only grab legend handles once, from row 0's sunny plots
        )

    # --- Cloudy row i -> columns 3,4,5 ---
    if i < n_cloudy:
        date = cloudy_dates_test[i]
        for k in range(3):
            ax = axarr[i, k + 3]
            ax.axis("on")
            plot_single_horizon(ax, date, "Cloudy", k, show_legend=False)

# Row-level y-axis labels (leftmost visible cell per row)
for r in range(n_rows):
    if axarr[r, 0].get_visible():
        axarr[r, 0].set_ylabel("PV (kW)", fontsize=8)

# Bottom-most populated cell per column gets an x-axis label
for c in range(6):
    for r in range(n_rows - 1, -1, -1):
        if axarr[r, c].has_data():
            axarr[r, c].set_xlabel("Hour of Day", fontsize=8)
            break

# Shared legend across the whole figure
handles, labels = [], []
for ax in axarr.flat:
    h, l = ax.get_legend_handles_labels()
    for hh, ll in zip(h, l):
        if ll and ll not in labels:
            handles.append(hh)
            labels.append(ll)
fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.01),
    ncol=4,
    frameon=True,
)

plt.tight_layout()
plt.show()

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "model_output/CNN_LSTM_forecast_15m_30m_1h_2017_2019_data/tensorboard_logs" 